# 1. Environment & Imports

โหลด autoreload สำหรับให้ Notebook รับโค้ดใหม่จาก `src/` อัตโนมัติเมื่อแก้ไข และ import `TextCleaner` ซึ่งเป็น Pipeline หลักสำหรับทำความสะอาดข้อมูล

In [20]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
import sys
import os

import pandas as pd

sys.path.append(os.path.abspath('../src'))

from data_cleaning import TextCleaner


# 2. Data Loading & Inspection

โหลดข้อมูลตัวอย่าง (`sample.csv`, ไม่มี header) และตรวจสอบโครงสร้าง/สัดส่วน Missing Values ของข้อมูลดิบก่อนเข้าสู่ขั้นตอนทำความสะอาด

In [22]:
raw_data = pd.read_csv(
    '../data/sample/sample.csv',
    header=None,
    index_col=False,
)
raw_data.head()


,0,1,2
0,1,Expensive Junk,This product consists of a piece of thin flexi...
1,1,Toast too dark,"Even on the lowest setting, the toast is too d..."
2,2,Excellent imagery...dumbed down story,I enjoyed this disc. The video is stunning. I ...
3,1,Are we pretending everyone is married?,The authors pretend that parents neither die n...
4,1,Not worth your time,"Might as well just use a knife, this product h..."


In [23]:
raw_data.info()


<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   0       50000 non-null  int64
 1   1       49997 non-null  str  
 2   2       50000 non-null  str  
dtypes: int64(1), str(2)
memory usage: 1.1 MB


# 3. Data Cleaning Pipeline

รัน Pipeline หลักจาก `TextCleaner` (`src/data_cleaning.py`): rename/map คอลัมน์, แปลง dtype, ล้างข้อความ (HTML/URL, whitespace, contractions), รวม `title` + `text` เป็นคอลัมน์ `review`, และกรอง duplicate/ข้อความสั้นเกินไปออก

In [24]:
cleaner = TextCleaner()
cleaned_data = cleaner.processing(raw_data)


Removed 0 duplicate reviews
Removed 0 empty/short reviews (< 3 chars)
Rows: 50000 -> 50000
Missing values per column:
sentiment    0
title        0
text         0
review       0
dtype: int64
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   sentiment  50000 non-null  category
 1   title      50000 non-null  str     
 2   text       50000 non-null  str     
 3   review     50000 non-null  str     
dtypes: category(1), str(3)
memory usage: 48.3 MB


# 4. Data Quality Check & Results

ตรวจผลลัพธ์หลังทำความสะอาด — Validation Report (จำนวนแถวก่อน/หลัง, duplicate ที่ถูกลบ, Missing Values, Memory Usage) ถูก print ไว้แล้วในขั้นตอนก่อนหน้าโดย `processing()`; cell นี้ใช้ตรวจตัวอย่างผลลัพธ์แบบ Interactive เพิ่มเติม

In [25]:
cleaned_data.head()


,sentiment,title,text,review
0,negative,expensive junk,this product consists of a piece of thin flexi...,expensive junk this product consists of a piec...
1,negative,toast too dark,"even on the lowest setting, the toast is too d...","toast too dark even on the lowest setting, the..."
2,positive,excellent imagery...dumbed down story,i enjoyed this disc. the video is stunning. i ...,excellent imagery...dumbed down story i enjoye...
3,negative,are we pretending everyone is married?,the authors pretend that parents neither die n...,are we pretending everyone is married? the aut...
4,negative,not worth your time,"might as well just use a knife, this product h...",not worth your time might as well just use a k...


ตรวจว่ายังมีคำที่มี apostrophe contraction (เช่น `n't`) หลงเหลืออยู่ในคอลัมน์ `review` หลังผ่าน Cleaning Pipeline หรือไม่ — ใช้เป็น Diagnostic เพื่อดูช่องว่าง ของการ Normalize คำย่อ/คำสะกดผิดที่ยังต้องปรับปรุงต่อ

In [26]:
cleaned_data[cleaned_data['review'].str.contains("n't")]

,sentiment,title,text,review
45,negative,item doen't work,this ups never worked from day one when i plug...,item doen't work this ups never worked from da...
1129,positive,hard to say if it is the 2nd or 3rd best,"if you really look at it, ""adventures on file ...",hard to say if it is the 2nd or 3rd best if yo...
1856,negative,too slow,i wanted to return it because the water is onl...,too slow i wanted to return it because the wat...
2853,negative,what a waste!!,a complete waste of money!! the earpieces did ...,what a waste!! a complete waste of money!! the...
3029,positive,incredible!,the author is very clear in his teachings. he ...,incredible! the author is very clear in his te...
...,...,...,...,...
47800,positive,this will send tingles down your spine.,this is quite possibly the best cd i have hear...,this will send tingles down your spine. this i...
47821,negative,not-so-sparkly,"my sister,lene, recieved this doll for her 6th...","not-so-sparkly my sister,lene, recieved this d..."
48362,positive,brandon peters,this 28'typeii- 225 pound rated werner is a ni...,brandon peters this 28'typeii- 225 pound rated...
48978,negative,kind of a drag,this wan't the tracy i was hoping for. the who...,kind of a drag this wan't the tracy i was hopi...


สังเกตว่าจำนวนแถวข้างบนไม่ได้ลดลงจากการเพิ่ม ftfy เพราะ regex นี้ค้นหาเฉพาะ apostrophe แบบตรง (`'`) แต่ ftfy แก้ปัญหาคนละจุด: คำย่อที่พิมพ์ด้วย apostrophe โค้ง/มน (`’`, เช่น `don’t`, `I’m`) ซึ่งไม่เคยถูกตรวจพบทั้งใน regex นี้และใน `contractions.fix()` เลยตั้งแต่แรก ตัวอย่างด้านล่างแสดงให้เห็นความแตกต่างก่อน/หลังเพิ่ม `fix_encoding()`

In [27]:
raw_curly_example = "So I\u2019m short about 19 0r so words for your review."
print('RAW           :', raw_curly_example)
print('CLEANED (now) :', cleaner.clean_text(pd.Series([raw_curly_example]))[0])


RAW           : So I’m short about 19 0r so words for your review.
CLEANED (now) : so i am short about 19 0r so words for your review.


# 5. Tokenization

ทดสอบ `TextProcessing.tokenization()` (`src/preprocessing.py`) กับ `cleaned_data` เต็มไฟล์ (`sample.csv`, 50,000 แถวหลัง cleaning) — ใช้ `spacy.blank('en')` (tokenizer อย่างเดียว ไม่มี tagger/parser/ner/lemmatizer) ผ่าน `nlp.pipe` แบบ batch แทนการ loop ทีละแถว เพื่อดู throughput และผลลัพธ์คอลัมน์ `tokens` ที่ได้ ก่อนต่อยอดไปทำ stopword removal / lemmatization

In [28]:
from preprocessing import TextProcessing

processor = TextProcessing()


In [ ]:
%%time
tokenized_data = processor.tokenization(cleaned_data)


CPU times: user 4.94 s, sys: 111 ms, total: 5.05 s
Wall time: 5.06 s


In [30]:
tokenized_data[['review', 'tokens']].head()


,review,tokens
0,expensive junk this product consists of a piec...,"[expensive, junk, this, product, consists, of,..."
1,"toast too dark even on the lowest setting, the...","[toast, too, dark, even, on, the, lowest, sett..."
2,excellent imagery...dumbed down story i enjoye...,"[excellent, imagery, ..., dumbed, down, story,..."
3,are we pretending everyone is married? the aut...,"[are, we, pretending, everyone, is, married, ?..."
4,not worth your time might as well just use a k...,"[not, worth, your, time, might, as, well, just..."


In [31]:
tokenized_data.info(memory_usage='deep')


<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   sentiment  50000 non-null  category
 1   title      50000 non-null  str     
 2   text       50000 non-null  str     
 3   review     50000 non-null  str     
 4   tokens     50000 non-null  object  
dtypes: category(1), object(1), str(3)
memory usage: 88.9 MB
